In [ ]:
pip install llama-index llama-index-utils-workflow llama-index-llms-google-genai llama-index-tools-google google-genai markdown-pdf httpx nest_asyncio

In [ ]:
import os
import getpass

print("="*50)
print("🛠️  FASE 1: CONFIGURACIÓN Y CONTEXTO")
print("="*50)

# 1. Forzar la petición de la API Key, ignorando la memoria de Colab
print("🔑 Ingresa tu Gemini API Key (Se ocultará mientras escribes):")
os.environ["GEMINI_API_KEY"] = getpass.getpass()
print("✅ API Key guardada en memoria.\n")

🛠️  FASE 1: CONFIGURACIÓN Y CONTEXTO
🔑 Ingresa tu Gemini API Key (Se ocultará mientras escribes):
··········
✅ API Key guardada en memoria.



In [ ]:
# 2. Pedir el PDF
pdf_path = input("📄 Ingresa el nombre de tu PDF (ej. semana7.pdf) o presiona Enter si no hay: ").strip()

📄 Ingresa el nombre de tu PDF (ej. semana7.pdf) o presiona Enter si no hay: semana7.pdf


In [ ]:
# 3. Pedir la instrucción
user_query = input("💬 Escribe tu petición para los agentes:\n> ").strip()

💬 Escribe tu petición para los agentes:
> Dame un resumen simplificado del tema del PDF


In [ ]:
import asyncio
import pathlib
import nest_asyncio
import traceback
import sys
from markdown_pdf import MarkdownPdf, Section

# LlamaIndex & Google GenAI
from google import genai
from google.genai import types as genai_types
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import (
    FunctionAgent,
    AgentWorkflow,
    AgentOutput,
    ToolCall,
    ToolCallResult
)

# Parche obligatorio para Colab
nest_asyncio.apply()

# ==========================================
# 1. CONFIGURACIÓN DE MODELOS (Optimizados para Capa Gratuita)
# ==========================================
# Cambiamos el Supervisor de 'pro' a 'flash' para evitar el Error 429 de cuota
llm_supervisor = GoogleGenAI(model="gemini-2.5-flash", temperature=0.0)
llm_researcher = GoogleGenAI(model="gemini-2.5-flash", temperature=0.0)
llm_writer = GoogleGenAI(model="gemini-2.5-flash", temperature=0.0)

# ==========================================
# 2. DEFINICIÓN DE HERRAMIENTAS
# ==========================================
google_search_tool = genai_types.Tool(google_search=genai_types.GoogleSearch())
llm_researcher_with_search = GoogleGenAI(
    model="gemini-3-flash-preview",
    generation_config=genai_types.GenerateContentConfig(tools=[google_search_tool])
)

async def search_web(ctx: Context, query: str) -> str:
    """Busca en Google información actualizada."""
    try:
        response = await llm_researcher_with_search.acomplete(
            f"Search Google and summarize the findings: {query}"
        )
        return str(response)
    except Exception as e:
        return f"Error en búsqueda web: {str(e)}"

# Nota: Volví a poner 'filepath' como parámetro para que el LLM sepa qué archivo lee
async def read_local_pdf(ctx: Context, filepath: str, query: str) -> str:
    """Extrae información de un archivo PDF local usando Gemini."""
    client = genai.Client()
    path = pathlib.Path(filepath)

    if not path.exists():
        return f"ERROR CRÍTICO: El archivo '{filepath}' no existe en la carpeta actual de Colab."

    try:
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=[
                genai_types.Part.from_bytes(data=path.read_bytes(), mime_type='application/pdf'),
                f"Extract and answer the following based ONLY on this document: {query}"
            ]
        )
        return response.text
    except Exception as e:
        return f"Error leyendo PDF: {str(e)}"

async def generate_final_pdf(ctx: Context, markdown_content: str, filename: str = "Respuesta_IA.pdf") -> str:
    """Convierte texto Markdown en un archivo PDF."""
    try:
        pdf = MarkdownPdf(toc_level=2)
        pdf.add_section(Section(markdown_content))
        pdf.save(filename)
        return f"SUCCESS! PDF guardado como {filename}."
    except Exception as e:
        return f"ERROR generando PDF: {str(e)}"

# ==========================================
# 3. AGENTES CON PROMPTS ESTRICTOS
# ==========================================
research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Lee PDFs locales y busca en internet.",
    system_prompt=(
        "You are the ResearchAgent. Your task is to gather information."
        "1. Use 'read_local_pdf' to read the document provided by the Supervisor."
        "2. Use 'search_web' to search for missing context online."
        "Once you have all the information, you MUST use the handoff tool to send the data to WriterAgent."
    ),
    llm=llm_researcher,
    tools=[search_web, read_local_pdf],
    can_handoff_to=["WriterAgent"] # Pasa directamente al redactor para evitar bucles infinitos
)

writer_agent = FunctionAgent(
    name="WriterAgent",
    description="Escribe el reporte y genera el PDF.",
    system_prompt=(
        "You are the WriterAgent. Format the data into a Markdown report."
        "You MUST use 'generate_final_pdf' to save it."
        "Only when the PDF is generated, hand off to SupervisorAgent."
    ),
    llm=llm_writer,
    tools=[generate_final_pdf],
    can_handoff_to=["SupervisorAgent"]
)

supervisor_agent = FunctionAgent(
    name="SupervisorAgent",
    description="Inicia el flujo delegando la tarea.",
    system_prompt=(
        "You are the Workflow Router. YOU CANNOT ANSWER THE USER."
        "You MUST immediately use the handoff tool to send the user's task to ResearchAgent."
        "Pass the user's exact instructions and the PDF file name to the ResearchAgent."
    ),
    llm=llm_supervisor,
    tools=[],
    can_handoff_to=["ResearchAgent"] # Solo puede mandarlo al investigador al inicio
)

workflow = AgentWorkflow(
    agents=[supervisor_agent, research_agent, writer_agent],
    root_agent="SupervisorAgent"
)

# ==========================================
# 4. MOTOR DE EJECUCIÓN (CON RAYOS X)
# ==========================================
async def main():
    print("\n" + "="*50)
    print("🚀 FASE 2: INICIANDO MOTOR AGÉNTICO")
    print("="*50)

    # Construimos un prompt inicial a prueba de balas
    instruccion_estricta = (
        f"INSTRUCCIÓN DEL USUARIO: '{user_query}'\n"
        f"ARCHIVO PDF A UTILIZAR: '{pdf_path}'\n\n"
        f"SUPERVISOR: DEBES delegar esto INMEDIATAMENTE al ResearchAgent. Dile que use la herramienta 'read_local_pdf' con el archivo '{pdf_path}'."
    )

    try:
        handler = workflow.run(user_msg=instruccion_estricta)
        current_agent = None

        async for event in handler.stream_events():
            # RAYOS X: Mostrar el evento crudo que dispara LlamaIndex
            print(f"\n[🔬 DEBUG INTERNO] Evento detectado: {type(event).__name__}")

            if hasattr(event, "current_agent_name") and event.current_agent_name != current_agent:
                current_agent = event.current_agent_name
                print(f"\n{'='*50}")
                print(f"🤖 AGENTE EN CONTROL: {current_agent}")
                print(f"{'='*50}")

            elif isinstance(event, AgentOutput):
                print(f"💬 [{current_agent}] Respuesta de Texto:")
                texto = event.response.content
                if texto:
                    print(f"   {texto}")
                else:
                    print("   [El agente no generó texto. Pasó directo a una herramienta o falló silenciosamente]")

                if event.tool_calls:
                    print(f"🧠 [{current_agent}] solicitó usar las herramientas: {[call.tool_name for call in event.tool_calls]}")
                elif not texto and not event.tool_calls:
                     print("🚨 ALERTA: El agente devolvió una respuesta completamente vacía y sin herramientas. Esto cerrará el flujo.")

            elif isinstance(event, ToolCall):
                print(f"⚙️  EJECUTANDO HERRAMIENTA: {event.tool_name}")
                print(f"   Argumentos: {event.tool_kwargs}")

            elif isinstance(event, ToolCallResult):
                print(f"✅ RESULTADO DE HERRAMIENTA ({event.tool_name}):")
                output_str = str(event.tool_output)
                print(f"   {output_str[:400]}... [Truncado]")

        print("\n" + "="*50)
        print("🏁 Fin del ciclo del flujo de trabajo.")
        print("="*50)

    except Exception as e:
        print("\n🚨 ERROR FATAL DETECTADO DURANTE LA EJECUCIÓN:")
        traceback.print_exc()

# Ejecutar
await main()


🚀 FASE 2: INICIANDO MOTOR AGÉNTICO

[🔬 DEBUG INTERNO] Evento detectado: AgentInput

🤖 AGENTE EN CONTROL: SupervisorAgent

[🔬 DEBUG INTERNO] Evento detectado: AgentStream

[🔬 DEBUG INTERNO] Evento detectado: AgentOutput
💬 [SupervisorAgent] Respuesta de Texto:
   [El agente no generó texto. Pasó directo a una herramienta o falló silenciosamente]
🧠 [SupervisorAgent] solicitó usar las herramientas: ['handoff']

[🔬 DEBUG INTERNO] Evento detectado: ToolCall
⚙️  EJECUTANDO HERRAMIENTA: handoff
   Argumentos: {'reason': "El usuario quiere un resumen simplificado del tema del PDF. Por favor, utiliza la herramienta 'read_local_pdf' con el archivo 'semana7.pdf' para procesar el PDF y luego proporciona el resumen.", 'to_agent': 'ResearchAgent'}

[🔬 DEBUG INTERNO] Evento detectado: ToolCallResult
✅ RESULTADO DE HERRAMIENTA (handoff):
   Agent ResearchAgent is now handling the request due to the following reason: El usuario quiere un resumen simplificado del tema del PDF. Por favor, utiliza la herr